#### setup

In [ ]:
import json
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

from library.data_utils import *
from library.models import *
from library.training import *
from library.evaluation import *

In [ ]:
# device and seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(seed=0)

In [ ]:
# parameters
dataset = 'synsum'
train_frac = 0.8

#### data

In [ ]:
# load data and split
df = pd.read_csv(f"./data/datasets/{dataset}.csv", index_col=0)
train_df, val_df, test_df = make_splits(df=df, train_frac=train_frac, seed=0)

In [ ]:
# load embeddings
embeddings = torch.load(f"./data/embeddings/{dataset}.pt", weights_only=True)

In [ ]:
# set confounders
confounders_text = ['dysp', 'cough', 'pain', 'nasal', 'fever_none', 'fever_low', 'fever_high']
confounders_tabular = ['self_empl', 'asthma', 'smoking', 'COPD', 'winter','hay_fever']
confounders_full = confounders_text + confounders_tabular

#### train risk-based predictive model

In [ ]:
# set parameters
input_dim = len(confounders_tabular) + 768
params = dict(
    hidden_dim=64,
    learning_rate=5e-4,
    weight_decay=0,
    batch_size=128,
    max_epochs=50,
    patience=5)

In [ ]:
# init data loaders
train_control = train_df[train_df["T"] == 0]
val_control   = val_df[val_df["T"] == 0]
train_loader, val_loader = make_predictive_loaders(train_control, val_control, confounders_tabular, embeddings, params['batch_size'])

# train predictive model
predictive_model = RegressionHead(input_dim=input_dim, hidden_dim=params["hidden_dim"]).to(device)
predictive_model, info = train_predictive(predictive_model, train_loader, val_loader, device, lr=params["learning_rate"], weight_decay=params["weight_decay"], 
                                          max_epochs=params["max_epochs"], patience=params["patience"], seed=0)

# store
torch.save(predictive_model.state_dict(), './data/checkpoints/predictive_model.pt')

#### evaluation

In [ ]:
# set model to eval
predictive_model.eval()

# init collectors
estimates, cates, M0s, M1s = [], [], [], []

# init test loader
test_loader = DataLoader(EvalDataset(test_df, confounders_tabular, confounders_full, embeddings), batch_size=1024, shuffle=False)

In [ ]:
# loop over test loader
with torch.inference_mode():
    for phi, x, cate, M0, M1 in test_loader:

        # forward pass
        phi = phi.to(device)
        risk = predictive_model(phi).squeeze(-1)

        # collect
        estimates.append(risk)
        cates.append(cate.squeeze(-1))
        M0s.append(M0.squeeze(-1))
        M1s.append(M1.squeeze(-1))

# store
to_np = lambda parts: torch.cat(parts, dim=0).detach().cpu().numpy()
df_eval = pd.DataFrame({"est": to_np(estimates), "cate": to_np(cates), "M0": to_np(M0s), "M1": to_np(M1s)})

In [ ]:
# sort and eval
ranked = df_eval.sort_values('est', ascending=False).copy()
print(f"PEHE          : {pehe(ranked):.6f}")
print(f"Policy value  : {policy_value(ranked):.6f}")
print(f"AUTOC         : {autoc(ranked):.6f}")